# Demo - Enron e-mail in, synthetic e-mail out

Runs the prototype end-to-end on real Enron e-mails.

**Pipeline** (`src/enron_synth/`): `parse` -> **stage 1: deterministic de-identification** (people, addresses, phones, ids, dates, Enron-isms) -> **stage 2: constrained LLM domain transfer** (OpenRouter) -> **verify + repair loop** -> render as a realistic raw e-mail.

**Offline / online.** Every LLM response is cached in `data/cache/`. With `OFFLINE = True` (default here) the notebook serves cached responses only and never touches the network, so it reproduces exactly and is free. Set `OFFLINE = False` (and put `OPENROUTER_API_KEY` in `.env`) to synthesise *new* e-mails - e.g. section 4.

In [1]:
import os, sys, warnings
OFFLINE = True
if OFFLINE:
    os.environ["SYNTH_OFFLINE"] = "1"
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
import pandas as pd
from enron_synth.pipeline import SyntheticEmailGenerator
from enron_synth.parsing import parse_email
from enron_synth.pii import Scrubber
from enron_synth.personas import COMPANY_BY_ID, PersonaFactory, COMPANIES
from enron_synth.llm import LLM, CacheMiss

gen = SyntheticEmailGenerator()          # default model: openai/gpt-4.1-mini via OpenRouter; default targets = 12 Indian companies
from enron_synth.personas import INTERNATIONAL_COMPANIES
gen_stored = SyntheticEmailGenerator(companies=INTERNATIONAL_COMPANIES)   # the stored LLM outputs (sections 2-3) were made with the earlier multi-country list
print("model:", gen.model, "| target companies:", [c.id for c in COMPANIES])

def synth(raw, file="", stored=False, **kw):
    try:
        return (gen_stored if stored else gen).generate(raw, file, **kw)
    except CacheMiss:
        print("(not in the response cache - set OFFLINE = False and provide an OpenRouter key to generate it)")

model: openai/gpt-4.1-mini | target companies: ['agriindia', 'bharaturja', 'kaveribank', 'sahyadri', 'tiranga', 'arogya', 'nilgiri', 'annapurna', 'himal', 'gangautil', 'iyerkapoor', 'deccan']


## 1. The example e-mail from the assessment brief
Raw input, then **stage 1 only** - exactly what the LLM is allowed to see. People, addresses, phones, numbers and dates are already synthetic; organisations, places and the company name are `[PLACEHOLDERS]` that stage 2 must invent. The LLM never receives the original people.

In [2]:
raw = open("../tests/sample_email.txt").read()
print(raw)

Message-ID: <1234.5678.JavaMail.evans@thyme>
Date: Thu, 7 Jun 2001 07:48:00 -0700 (PDT)
From: william.giuliani@enron.com
To: andrew.fastow@enron.com
Cc: mark.haedicke@enron.com
Subject: Approval of the DPR Accelerated Put transaction
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
X-From: William Giuliani
X-To: Andrew S Fastow <Andrew S Fastow/HOU/ECT@ECT>
X-cc: Mark E Haedicke <Mark E Haedicke/HOU/ECT@ECT>
X-Folder: \Andrew_Fastow_Dec2001\Notes Folders\Sent
X-Origin: Fastow-A

Dear Andrew,

Attached is the DASH for the approval of the DPR Accelerated Put transaction. This partial divestiture allows us to put $11 million of our equity interest back to DPR Holding Company, LLC, and its subsidiary, Dakota, LLC. Both entities are controlled by Chris Cline.

In addition to redeeming part of our equity interest, the deal provides us with 900,000 tons of coal priced below market, an option which could lead to a very profitable synfuel project, and the potential for more marketin

In [3]:
e = parse_email(raw, "fastow-a/sent/1.")
s = Scrubber(PersonaFactory(COMPANY_BY_ID["agriindia"])).scrub(e, shift_days=8400)
print("SUBJECT:", s.subject, "\n"); print(s.body); print("\nreplacement counts:", s.counts)

SUBJECT: Approval of the DPR [ORG_1] transaction 

Dear Balhaar,

Attached is the DASH for the approval of the DPR [ORG_1] transaction. This partial divestiture allows us to put $11 million of our equity interest back to [ORG_2], LLC, and its subsidiary, [LOC_1], LLC. Both entities are controlled by Charles Dasgupta.

In addition to redeeming part of our equity interest, the deal provides us with 900,000 tons of coal priced below market, an option which could lead to a very profitable synfuel project, and the potential for more marketing fees from other Dasgupta entities.

The DASH has been approved and signed by RAC and [ORG_4] and is now awaiting Gabriel Nazareth's review and approval. I wanted to give you the opportunity to review the DASH and become familiar with the provisions of the deal.

If you have any questions on the transaction, feel free to contact me at +91 98515 42387. Others familiar with the deal are Advaith Tiwari, Tanish Bumb, and Christopher Chauhan.

Thank you.

Be

Baseline **rule mode** (no LLM: placeholders filled from Faker) shows what de-identification alone gives - the business content is still Enron's:

In [4]:
print(gen.generate(raw, "fastow-a/sent/1.", company="kaveribank", mode="rule").raw)

Message-ID: <88d90260daea7211.85860@kaveribank.co.in>
Date: Thu, 06 Jun 2024 07:48:00 +0530
From: Oscar Sekhon <oscar.sekhon@kaveribank.co.in>
To: Jeet Gaba <jeet.gaba@kaveribank.co.in>
Cc: Faqid Sami <faqid.sami@kaveribank.co.in>
Subject: Approval of the DPR Rege-Bassi transaction
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii

Dear Jeet,

Attached is the DASH for the approval of the DPR Rege-Bassi transaction. This partial divestiture allows us to put $11 million of our equity interest back to Vig-Mangat, LLC, and its subsidiary, Bally, LLC. Both entities are controlled by Hritik Biswas.

In addition to redeeming part of our equity interest, the deal provides us with 900,000 tons of coal priced below market, an option which could lead to a very profitable synfuel project, and the potential for more marketing fees from other Biswas entities.

The DASH has been approved and signed by RAC and Barman, Iyengar and Mutti and is now awaiting Faqid Sami's review and approval. I

**Full hybrid pipeline** on the same e-mail (needs the API the first time; cached afterwards):

In [5]:
o = synth(raw, "fastow-a/sent/1.", company="agriindia")
if o: print(o.raw); print("\nmeta:", {k: o.meta[k] for k in ["model", "attempts", "leaks_after", "forced_redaction"]})

(not in the response cache - set OFFLINE = False and provide an OpenRouter key to generate it)


**Recorded output** of that call from an earlier session (pipeline v1, `openai/gpt-4.1-mini`, target = Agriculture India). The Enron text is the brief's example; the output below was produced by the pipeline, not written by hand:

```
Message-ID: <caee4903d563ef0c.58556@agriindia.com>
Date: Thu, 06 Jun 2024 07:48:00 +0530
From: Advik Kara <advik.kara@agriindia.com>
To: Balhaar Sanghvi <balhaar.sanghvi@agriindia.com>
Subject: Approval of the DPR Kaveri Commodities transaction
Cc: Gabriel Nazareth <gabriel.nazareth@agriindia.com>
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii

Dear Balhaar,

Attached is the DASH for the approval of the DPR Kaveri Commodities transaction. This partial divestiture allows us to return ₹8.2 crore of our equity interest back to Aryan Grains, Pvt. Ltd., and its subsidiary, Pune Agro Holdings. Both entities are controlled by Charles Dasgupta.

In addition to redeeming part of our equity interest, the deal provides us with 900,000 quintals of wheat priced below market, an option which could lead to a very profitable flour milling project, and the potential for more brokerage fees from other Dasgupta-affiliated firms.

The DASH has been approved and signed by RAC and AgriIndia's Farm Partnerships unit and is now awaiting Gabriel Nazareth's review and approval. I wanted to give you the opportunity to review the DASH and become familiar with the provisions of the deal.

If you have any questions on the transaction, feel free to contact me at +91 98515 42387. Others familiar with the deal are Advaith Tiwari, Tanish Bumb, and Christopher Chauhan.

Thank you.

Best regards,
Advik Kara
Agriculture India Pvt. Ltd., Pune
```

Note what v1 still got wrong (and v2 fixes by instructing the LLM to replace internal jargon): the Enron-internal terms **DASH** and **RAC** survived. Everything identifying (people, company, counterparties, amounts, phone, city) was replaced consistently, in the domain of the target company.

## 2. Random e-mails from the evaluation set (stored LLM outputs from the earlier multi-country run)
Original vs. synthetic: hard-wrapping, forwarded history, tone and structure are preserved; people, companies, places, numbers, dates and currency are not.

In [6]:
ev = pd.read_json("../data/eval_set.jsonl", lines=True)
def show(i, n=1300):
    r = ev.iloc[i]
    body = r["message"].split("X-FileName:")[1].split("\n", 1)[1]
    o = synth(r["message"], r["file"], stored=True, seed=str(i))
    print("#" * 100, f"\nORIGINAL  ({r['file']})\n" + body[:n], "\n\n----------------- SYNTHETIC -----------------\n")
    if o:
        print(o.raw[:n + 350]); print("\nmeta:", {"company": o.company} | {k: o.meta[k] for k in ["attempts", "leaks_after", "forced_redaction"]})
for i in [3, 11, 20]:
    show(i)

#################################################################################################### 
ORIGINAL  (kaminski-v/sent/1695.)

Shelly,

These are the Super Saturdays I can help you with:

Nov 10
Dec 1
Dec 8
Dec 15 

I shall be traveling on two other days. One of these trips is related to 
recruiting at CMU.

Vince 

----------------- SYNTHETIC -----------------

Message-ID: <8beb1df7edbb7989.43036@lumieresante.fr>
Date: Wed, 22 Oct 2025 10:13:00 +0200
From: Aimé Ruiz <aime.ruiz@lumieresante.fr>
To: Danielle Jourdan <danielle.jourdan@lumieresante.fr>
Cc: Aimé Ruiz <aime.ruiz@lumieresante.fr>, Laetitia Charles <laetitia.charles@lumieresante.fr>
Subject: Super Saturday
Mime-Version: 1.0
Content-Type: text/plain; charset=utf-8

Danielle,

These are the Lyon clinics I can support you with:

7 Nov
28 Nov
5 Dec
12 Dec

I'll be away on two other dates. One trip involves
interviews at Saint-Luc Medical Center.

Aimé

meta: {'company': 'lumiere', 'attempts': 1, 'leaks_after': [], 'forc

#################################################################################################### 
ORIGINAL  (martin-t/inbox/132.)

Tom,

Thanks for the info.

What dates are you associating with Phase 1 and Phase 2?

Dave

 -----Original Message-----
From: 	Martin, Thomas A.  
Sent:	Tuesday, January 08, 2002 10:12 AM
To:	Forster, David
Cc:	Winfree, O'Neal D.; Redmond, Brian
Subject:	Texas Desk EOL Product Rollout Schedule


As a follow up to the Texas Business plan submitted on January 7th, please find more detail on our desired product rollout schedule during phase 1 and 2.  If you have any questions call me at ext. 33079.

Tom


 << File: TXEOLRollout.xls >>  

----------------- SYNTHETIC -----------------

Message-ID: <a815340d5de7d2fe.46778@lakeshoreutilities.gov>
Date: Tue, 09 Jan 2024 08:27:03 -0500
From: Charles Obrien <charles.obrien@lakeshoreutilities.gov>
Subject: Re: Riverview Water Main Replacement Schedule
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii

A

## 3. Consistency
Same Enron person -> same synthetic person inside a mailbox (threads stay coherent); different mailbox -> different company. Names below are the synthetic senders for one Enron mailbox:

In [7]:
rows = []
for i in range(len(ev)):
    r = ev.iloc[i]
    if r["file"].startswith("kaminski-v"):
        try:
            o = gen_stored.generate(r["message"], r["file"], seed=str(i))
        except CacheMiss:
            continue
        rows.append((o.company, o.headers["From"], o.headers["To"][:60], o.headers["Date"]))
pd.DataFrame(rows, columns=["company", "From", "To", "Date"]).head(8)

,company,From,To,Date
0,lumiere,Aimé Ruiz <aime.ruiz@lumieresante.fr>,Danielle Jourdan <danielle.jourdan@lumieresant...,"Wed, 22 Oct 2025 10:13:00 +0200"
1,lumiere,Thibaut Pons <thibaut.pons@lumieresante.fr>,Aimé Ruiz <aime.ruiz@lumieresante.fr>,"Tue, 21 Jan 2025 02:33:00 +0100"
2,lumiere,Aimé Ruiz <aime.ruiz@lumieresante.fr>,Pierre Meyer <pierre.meyer@lumieresante.fr>,"Wed, 06 Aug 2025 06:00:00 +0200"
3,lumiere,Alexandre Bonnin <alexandre.bonnin@lumieresant...,René Bourgeois <rene.bourgeois@lumieresante.fr>,"Tue, 04 Nov 2025 06:26:02 +0100"


## 4. Your own e-mail
Paste any raw Enron message (headers + body). Needs `OFFLINE = False` + an API key the first time. CLI equivalent: `python -m enron_synth.cli one --input my_email.txt`

In [8]:
my_email = '''Message-ID: <1.2.JavaMail.evans@thyme>
Date: Tue, 3 Oct 2000 09:12:00 -0700 (PDT)
From: sara.shackleton@enron.com
To: mark.taylor@enron.com
Subject: Re: ISDA master - Credit Suisse
X-From: Sara Shackleton
X-To: Mark E Taylor

Mark,

Please find attached the revised ISDA schedule for Credit Suisse First Boston. I have flagged the two provisions that the credit group wants to negotiate. This is privileged and confidential - do not forward outside legal.

Let's talk Thursday morning.

Sara
'''
o = synth(my_email, "shackleton-s/sent/1.")
if o: print(o.raw)
print(LLM.report())

(not in the response cache - set OFFLINE = False and provide an OpenRouter key to generate it)
LLM calls=0 cache_hits=11 spent=$0.0000
